# Анализ погодных данных (финал пайплайна)

Пайплайн: **генератор → ClickHouse → Jupyter**. В этом ноутбуке подключаемся к ClickHouse, смотрим базовую статистику и строим пару графиков по показаниям метеостанций.

In [ ]:
%matplotlib inline
import os
import clickhouse_connect
import pandas as pd
import matplotlib.pyplot as plt

# Jupyter работает в compose-сети: хост ClickHouse = имя сервиса.
client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "clickhouse"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username=os.environ.get("CLICKHOUSE_USER", "default"),
    password=os.environ.get("CLICKHOUSE_PASSWORD", "clickhouse"),
)
print("Подключено к ClickHouse")

In [ ]:
db = os.environ.get("CLICKHOUSE_DB", "weather")
rows = client.query(
    f"SELECT count(), countDistinct(station_id), min(timestamp), max(timestamp) FROM {db}.readings"
).result_rows[0]
print(f"Показаний: {rows[0]:,}".replace(",", " "))
print(f"Станций: {rows[1]}")
print(f"Диапазон: {rows[2]}  …  {rows[3]}")

In [ ]:
# Средняя температура по часам (24 ч от последнего замера в базе)
sql = f"""
SELECT toStartOfHour(timestamp) AS hour,
       round(avg(temperature_c), 2) AS avg_temp
FROM {db}.readings
WHERE timestamp >= (SELECT max(timestamp) FROM {db}.readings) - INTERVAL 24 HOUR
GROUP BY hour ORDER BY hour
"""
df = client.query_df(sql)
df["hour"] = pd.to_datetime(df["hour"])
df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["hour"], df["avg_temp"], marker="o", linewidth=2)
plt.title("Средняя температура по часам (последние 24 ч)")
plt.ylabel("°C")
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

In [ ]:
# Аномалии по дням
anom = client.query_df(f"""
    SELECT toDate(timestamp) AS day, sum(is_anomaly) AS anomalies
    FROM {db}.readings
    WHERE is_anomaly = 1
    GROUP BY day ORDER BY day
""")
anom["day"] = pd.to_datetime(anom["day"])

plt.figure(figsize=(12, 3))
plt.bar(anom["day"], anom["anomalies"])
plt.title("Аномалии по дням")
plt.ylabel("кол-во")
plt.tight_layout()
plt.show()